# Issue 006 Prompt #1 — Pipeline A VideoMAE Scaffold

**Status:** scaffold only. Data wiring, GPU recording, tiny-slice overfit, evaluation-harness plumbing. **Not** in this step: full fine-tune, real validation, scheduler, or weights pushed anywhere.

This notebook reads **already-computed crop shards** from Drive artefacts and **never re-runs** raw-frame staging, perception, or cropping. If those shards are missing, the notebook fails loud with a clear "run `colab/000_full_pipeline.ipynb` first" message.

Reuses the existing setup/layout modules instead of duplicating configuration:

- `colab/setup.py` (Drive layout, env lock, run log)
- `colab/data_mode.py` (LOCAL / DRIVE resolution)
- `pipeline_a/load_crops.py` (the Issue 005 Step 1 crop loader)
- `cropping/shard_writer.py` (read-only shard utilities — *not* the runner)
- `data/urfd_labels.py` (`label_window_from_crop_meta` — the supported real-crop entry point)
- `evaluation/metrics/classification.py` (the Issue 004 Step 2 metric bundle)

### Hard rules carried forward from Issues 001–005

- Detector was `yolo26m`, tracker was `bytetrack.yaml`. **Out of scope** for this notebook — Issue 002 produced the shards.
- Crop shards are 32 × 224 × 224 (URFD cam0 only), ImageNet-normalised by the loader. This notebook does NOT re-normalise.
- Train / validation split is **by source clip**, never by window.
- Frozen unseen tier is **inaccessible** from this scaffold — `ExecutionContext.EVALUATION` denies frozen access by design.
- The Hugging Face video-classification tutorial is an **API reference only** — its accuracy-only metric, `.avi` data pipeline, normalisation, and Hub-push flow are NOT copied.

### Why this is only Prompt #1

Per the Issue 006 spec, this notebook proves the wiring (model loads, dataset produces tensors of the right shape, loss can collapse on a tiny slice, eval plumbing produces real numbers). A long fine-tune run is a separate prompt.

## 1. Mount Drive + resolve layout

**LOCAL mode** is the default — datasets live on the Colab local disk during staging / perception / cropping. This scaffold reads only from Drive artefacts (shards, labels, logs), so Drive is the only thing that needs to be mounted.

Override via `FALL_DETECTION_DATA_MODE=local|drive` or by passing a different root. The notebook never stages anything from raw frames, so the dataset root is informational here.

In [ ]:
import os, pathlib, sys

# Mount Drive (or skip if not on Colab — local dev still works because the
# notebooks read from FALL_DETECTION_PROJECT_ROOT, not from /content/drive).
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not on Colab — Drive mount skipped. Set FALL_DETECTION_PROJECT_ROOT if running outside Colab.")

PROJECT_ROOT = pathlib.Path(os.environ.get("FALL_DETECTION_PROJECT_ROOT", "/content/fall_detection"))
if not PROJECT_ROOT.exists():
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from colab.data_mode import DataMode, resolve_data_layout, describe_layout

DATA_MODE = DataMode(os.environ.get("FALL_DETECTION_DATA_MODE", "local").lower())
layout = resolve_data_layout(mode=DATA_MODE)

print(f"Data mode        : {layout.mode.value}")
print(f"Dataset root     : {layout.dataset_root}")
print(f"Artefact root    : {layout.artifact_root}")
print(f"Layout summary   : {describe_layout(layout)}")

# Resolved artefact paths this notebook writes to.
CROPS_ROOT = layout.artifacts / "crops"
CSV_LABEL_DIR = layout.artifact_root / "artifacts" / "urfd_labels"
CKPT_DIR = layout.artifact_root / "checkpoints" / "pipeline_a"
METRICS_DIR = layout.artifact_root / "metrics" / "pipeline_a"
for d in (CROPS_ROOT, CSV_LABEL_DIR, CKPT_DIR, METRICS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Crops root       : {CROPS_ROOT}")
print(f"CSV label dir    : {CSV_LABEL_DIR}")
print(f"Checkpoint dir   : {CKPT_DIR}")
print(f"Metrics dir      : {METRICS_DIR}")


## 1a. GPU / environment recording

The Issue 006 carry-forwards call out: capability-detect fp16 / bf16 before training, record actual GPU + VRAM + CUDA + torch, and pin the run with a JSON run-log sidecar so Issue 015 (real-time opt) and Issue 004 (eval harness) have stable provenance.

Pure stdlib + torch / subprocess — no ultralytics, no transformers (those imports land in §3 where they're needed).

In [ ]:
import json, platform, shutil, subprocess
from datetime import datetime, timezone

import torch

def _query_nvidia_smi():
    """Best-effort GPU name + VRAM (GiB) via nvidia-smi."""
    if shutil.which("nvidia-smi") is None:
        return None, None
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total",
             "--format=csv,noheader,nounits"],
            text=True, timeout=10).strip()
    except Exception:
        return None, None
    if not out:
        return None, None
    first = out.splitlines()[0]
    parts = [p.strip() for p in first.split(",")]
    if len(parts) < 2:
        return parts[0], None
    try:
        return parts[0], round(float(parts[1]) / 1024.0, 2)
    except ValueError:
        return parts[0], None

gpu_name, vram_gib = _query_nvidia_smi()
torch_cuda = torch.version.cuda
bf16_supported = bool(
    torch.cuda.is_available() and torch.cuda.is_bf16_supported()
) if hasattr(torch.cuda, "is_bf16_supported") else False
fp16_supported = bool(torch.cuda.is_available())

env_record = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "gpu_name": gpu_name,
    "gpu_vram_gib": vram_gib,
    "torch_version": torch.__version__,
    "torch_cuda_version": torch_cuda,
    "torch_cuda_available": bool(torch.cuda.is_available()),
    "bf16_supported": bf16_supported,
    "fp16_supported": fp16_supported,
    "python_version": platform.python_version(),
    "data_mode": layout.mode.value,
    "crops_root": str(CROPS_ROOT),
    "csv_label_dir": str(CSV_LABEL_DIR),
}

print("Recorded environment:")
for k, v in env_record.items():
    print(f"  {k:<24} = {v}")

# Persist the run-log sidecar so Issue 015 / Issue 004 can correlate.
RUN_LOG_PATH = METRICS_DIR / "006_pipeline_a_run_log.json"
RUN_LOG_PATH.write_text(json.dumps(env_record, indent=2, sort_keys=True), encoding="utf-8")
print(f"\nRun-log sidecar: {RUN_LOG_PATH}")

# Loud fail if no CUDA — VideoMAE on CPU is not the path we are testing here.
assert torch.cuda.is_available(), (
    "CUDA is not available. Issue 006 requires a Colab GPU runtime; "
    "switch the runtime type (T4 / L4 / A100) and re-run."
)

## 2. Crop-availability guard

This notebook reads `shard-*.tar` from `CROPS_ROOT`. The expected current cache shape (verified at Issue 005 close-out) is **70 shards, 240 windows, 41 fall, 199 no-fall**. If the cache is missing or inconsistent, the notebook **fails loud with a clear "run `colab/000_full_pipeline.ipynb` first"** — never silently rebuilds from raw frames.

The check is purely a guard: it reads `list_clip_keys` + one metadata sidecar per clip key. It does NOT load pixels.

In [ ]:
from pipeline_a import REQUIRED_METADATA_FIELDS
from cropping.shard_writer import read_shard, safe_member_name
from data.urfd_labels import (
    parse_urfd_csv_label_file,
    label_window_from_crop_meta,
    WINDOW_LABEL_FALL,
    WINDOW_LABEL_NO_FALL,
    WINDOW_LABEL_UNLABELED,
)

FALL_CSV_FILENAME = "urfall-cam0-falls.csv"
ADL_CSV_FILENAME = "urfall-cam0-adls.csv"

EXPECTED_SHARDS_MIN = 65
EXPECTED_SHARDS_MAX = 75
EXPECTED_WINDOWS_MIN = 220
EXPECTED_WINDOWS_MAX = 260
EXPECTED_UNIQUE_CLIPS_MIN = 65
EXPECTED_UNIQUE_CLIPS_MAX = 75

shard_paths = sorted(CROPS_ROOT.glob("shard-*.tar"))
if not shard_paths:
    raise SystemExit(
        f"No crop shards found under {CROPS_ROOT}.\n"
        f"Run colab/000_full_pipeline.ipynb first to produce shards, then re-run this notebook.\n"
        f"(That notebook handles Drive mounting, URFD staging, perception, and crop generation.)"
    )

# Walk every shard; for each clip_key, collect the window's frame_indices
# (one per sidecar, 0-based — the same indexing the pipeline_a loader
# uses) plus its source-clip label from the sidecar.
#
# meta["label"] is treated as SOURCE CLIP provenance only — it is the
# `clip_record.label` from the Issue 003 cropping runner, NOT a
# window-level fall/no_fall decision. Window-level labels come from
# label_window_from_crop_meta(...) below, never from meta["label"].
records = []  # list of dicts: clip_key, clip_id, source_label, frame_indices
for sp in shard_paths:
    res = read_shard(sp)
    for clip_key in res.manifest.get("clip_keys", []):
        safe = safe_member_name(clip_key)
        per_window_members = {
            name: meta for name, meta in res.metadata_members.items()
            if name.startswith(safe + "_") and name.endswith(".meta.json")
        }
        if not per_window_members:
            raise SystemExit(
                f"Shard {sp.name} clip {clip_key!r}: no metadata sidecars found.\n"
                f"Re-run Issue 003 (colab/000_full_pipeline.ipynb) to regenerate shards."
            )
        # Required-fields check on every sidecar (cheap dict membership).
        for name, meta in per_window_members.items():
            for f in REQUIRED_METADATA_FIELDS:
                if f not in meta:
                    raise SystemExit(
                        f"Shard {sp.name} sidecar {name!r}: missing required field {f!r}."
                    )
        # All sidecars for a window agree on provenance — take the first.
        first_meta = next(iter(per_window_members.values()))
        clip_id = first_meta["clip_id"]
        source_label = first_meta["label"]
        frame_indices = sorted(
            int(m["frame_index"]) for m in per_window_members.values() if "frame_index" in m
        )
        records.append({
            "clip_key": clip_key,
            "clip_id": clip_id,
            "source_label": source_label,
            "frame_indices": frame_indices,
        })

n_shards = len(shard_paths)
n_windows = len(records)
n_source_fall_windows = sum(1 for r in records if r["source_label"] == "fall")
n_source_nofall_windows = sum(1 for r in records if r["source_label"] == "no_fall")
unique_clip_ids = sorted({r["clip_id"] for r in records})

print(f"Shards on Drive        : {n_shards}    (expected {EXPECTED_SHARDS_MIN}..{EXPECTED_SHARDS_MAX})")
print(f"Windows on Drive       : {n_windows}    (expected {EXPECTED_WINDOWS_MIN}..{EXPECTED_WINDOWS_MAX})")
print(f"  source-label fall    : {n_source_fall_windows}    (windows from FALL clips — source-clip provenance, NOT window-level labels)")
print(f"  source-label no_fall : {n_source_nofall_windows}    (windows from ADL clips — source-clip provenance, NOT window-level labels)")
print(f"Unique source clips    : {len(unique_clip_ids)}    (expected {EXPECTED_UNIQUE_CLIPS_MIN}..{EXPECTED_UNIQUE_CLIPS_MAX})")

fail_msg = []
if not (EXPECTED_SHARDS_MIN <= n_shards <= EXPECTED_SHARDS_MAX):
    fail_msg.append(f"shard count {n_shards} is outside {EXPECTED_SHARDS_MIN}..{EXPECTED_SHARDS_MAX}.")
if not (EXPECTED_WINDOWS_MIN <= n_windows <= EXPECTED_WINDOWS_MAX):
    fail_msg.append(f"window count {n_windows} is outside {EXPECTED_WINDOWS_MIN}..{EXPECTED_WINDOWS_MAX}.")
if not (EXPECTED_UNIQUE_CLIPS_MIN <= len(unique_clip_ids) <= EXPECTED_UNIQUE_CLIPS_MAX):
    fail_msg.append(f"unique-clip count {len(unique_clip_ids)} is outside {EXPECTED_UNIQUE_CLIPS_MIN}..{EXPECTED_UNIQUE_CLIPS_MAX}.")
# Every clip_id must be URFD debug family with a recognised sequence.
for r in records:
    ci = r["clip_id"]
    if not (ci.startswith("urfd-debug-") and ci.endswith("-cam0-rgb")):
        fail_msg.append(f"unexpected clip_id shape: {ci!r} (every clip must be urfd-debug-<seq>-cam0-rgb)")
        break

# CSV presence.
fall_csv_path = CSV_LABEL_DIR / FALL_CSV_FILENAME
adl_csv_path = CSV_LABEL_DIR / ADL_CSV_FILENAME
for p in (fall_csv_path, adl_csv_path):
    if not p.is_file():
        raise SystemExit(
            f"URFD label CSV missing: {p}.\n"
            f"Run colab/000_full_pipeline.ipynb first to download the CSVs (university source)."
        )

# Stronger correct check: window-level labels via label_window_from_crop_meta.
# Cheap (240 windows × ~32 frame lookups, sub-second total) and catches
# the most common wiring regressions — wrong CSV, wrong offset, swapped
# fall/ADL directory — without requiring an exact count match. The exact
# window-level fall count depends on which windows span the falling/
# lying region vs. the pre-fall region of each fall clip, so we only
# check consistency invariants:
#
#   1. Every window resolves to one of {fall, no_fall, unlabeled}.
#   2. Sum of fall + no_fall + unlabeled equals total windows.
#   3. ADL-source (no_fall source label) windows never resolve to "fall"
#      via the CSV — that would be a CSV / offset wiring bug.
fall_csv = parse_urfd_csv_label_file(fall_csv_path)
adl_csv = parse_urfd_csv_label_file(adl_csv_path)

n_window_fall = 0
n_window_nofall = 0
n_window_unlabeled = 0
lookup_errors = []
for r in records:
    csv = fall_csv if "fall-" in r["clip_id"] else adl_csv
    try:
        label_str, _ = label_window_from_crop_meta(
            r["clip_id"], list(r["frame_indices"]), csv,
        )
    except Exception as exc:
        lookup_errors.append((r["clip_key"], f"{type(exc).__name__}: {exc}"))
        continue
    if label_str == WINDOW_LABEL_FALL:
        n_window_fall += 1
    elif label_str == WINDOW_LABEL_NO_FALL:
        n_window_nofall += 1
    elif label_str == WINDOW_LABEL_UNLABELED:
        n_window_unlabeled += 1
    else:
        lookup_errors.append((r["clip_key"], f"unexpected label_str {label_str!r}"))

total_labelled = n_window_fall + n_window_nofall + n_window_unlabeled
if total_labelled != n_windows:
    fail_msg.append(
        f"window-level labels cover {total_labelled} of {n_windows} windows; "
        f"every window must resolve to fall/no_fall/unlabeled."
    )

# Belt-and-braces: ADL-source windows must NEVER resolve to "fall".
adl_fall_mismatch = 0
for r in records:
    if r["source_label"] != "no_fall":
        continue
    label_str, _ = label_window_from_crop_meta(
        r["clip_id"], list(r["frame_indices"]), adl_csv,
    )
    if label_str == WINDOW_LABEL_FALL:
        adl_fall_mismatch += 1
if adl_fall_mismatch > 0:
    fail_msg.append(
        f"{adl_fall_mismatch} ADL-source windows resolved to 'fall' — "
        f"likely a CSV / offset wiring bug (ADL clips cannot contain a fall)."
    )

if lookup_errors:
    fail_msg.append(
        f"{len(lookup_errors)} window(s) had label-lookup errors. First: {lookup_errors[0]}"
    )

if fail_msg:
    raise SystemExit(
        "Crop cache shape inconsistent with the Issue 005 close-out.\n"
        "Failures:\n  - " + "\n  - ".join(fail_msg) + "\n\n"
        "Fix: run colab/000_full_pipeline.ipynb first (with FORCE_RECOMPUTE=True "
        "if the shards are stale), then re-run this notebook."
    )

print()
print("Window-level labels (via label_window_from_crop_meta — NOT meta['label']):")
print(f"  fall       : {n_window_fall}    (windows spanning the falling/lying region of a fall clip)")
print(f"  no_fall    : {n_window_nofall}    (pre-fall windows of fall clips + every ADL window)")
print(f"  unlabeled  : {n_window_unlabeled}    (windows with zero available CSV labels)")
print()
print("Crop-availability guard: PASSED.")

# Compatibility alias for later cells that consume `windows`.
# `records` carries the full per-window info (clip_key, clip_id,
# source_label, frame_indices); `windows` is the (clip_key,
# clip_id, source_label) tuple the §6 split cell expects. Defining
# it here — at the end of §2 — keeps §6 free of any reference to
# `records` (the §6 cell consumes only what it needs).
windows = [(r["clip_key"], r["clip_id"], r["source_label"]) for r in records]

## 3. Model load — frozen Kinetics backbone with binary head

We use **`MCG-NJU/videomae-base-finetuned-kinetics`** as the primary backbone. The Kinetics-finetuned variant's backbone architecture is identical to plain `videomae-base`; the difference is the classifier head (700 Kinetics classes vs our 2). We re-initialise the head for binary classification; the backbone is loaded strictly.

Plain `MCG-NJU/videomae-base` is **kept as an optional ablation note** — switching to it would skip the Kinetics pre-training signal that the frozen backbone carries. The main path uses the Kinetics-finetuned weights.

A trust-check pass follows the load: the Kinetics-finetuned state dict is downloaded separately and every non-classifier key is asserted to match (shape + presence) the loaded model. The classifier head is allowed to differ because of `num_labels=2 + ignore_mismatched_sizes=True`; backbone mismatches are not allowed and fail loud.


In [ ]:
import safetensors.torch as st
from huggingface_hub import hf_hub_download
import transformers
from transformers import VideoMAEForVideoClassification, VideoMAEConfig

PRIMARY_MODEL_ID = "MCG-NJU/videomae-base-finetuned-kinetics"
ABLATION_MODEL_ID = "MCG-NJU/videomae-base"  # ablation note: not the main path

LABEL2ID = {"no_fall": 0, "fall": 1}
ID2LABEL = {0: "no_fall", 1: "fall"}

# Record installed versions in the run log. The transformers pin is
# enforced here (and in colab/setup.py:APPROVED_PIP_PACKAGES) so Colab
# cannot silently use a newer incompatible VideoMAE implementation.
print(f"transformers version     : {transformers.__version__}")
print(f"Primary model id         : {PRIMARY_MODEL_ID}")
assert transformers.__version__ == "4.46.3", (
    f"transformers version is {transformers.__version__}; this notebook is pinned to 4.46.3.\n"
    f"Re-run colab/000_full_pipeline.ipynb to re-install the approved stack."
)
print(f"Ablation model id (note) : {ABLATION_MODEL_ID}")

# Load the Kinetics-finetuned backbone with a 2-class head. The 700-class
# Kinetics head is dropped and re-initialised by from_pretrained via
# num_labels + label2id/id2label + ignore_mismatched_sizes=True.
model = VideoMAEForVideoClassification.from_pretrained(
    PRIMARY_MODEL_ID,
    num_labels=2,
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    ignore_mismatched_sizes=True,
)

model_cfg: VideoMAEConfig = model.config
MODEL_T = int(model_cfg.num_frames)
MODEL_H = int(model_cfg.image_size)
MODEL_W = int(model_cfg.image_size)
HIDDEN_SIZE = int(model_cfg.hidden_size)
TUBELET = int(model_cfg.tubelet_size)
PATCH = int(model_cfg.patch_size)

print(f"num_labels            : {model_cfg.num_labels}")
print(f"num_frames (T)        : {MODEL_T}     <- the model's expected temporal length")
print(f"image_size (H=W)      : {MODEL_H}    <- must match the loader's IMAGE_SIZE")
print(f"hidden_size           : {HIDDEN_SIZE}    <- backbone output dim (linear head input)")
print(f"tubelet_size          : {TUBELET}    <- 3D conv kernel on the time axis")
print(f"patch_size            : {PATCH}")
print(f"label2id              : {model_cfg.label2id}")
print(f"id2label              : {model_cfg.id2label}")

from pipeline_a import IMAGE_SIZE as LOADER_IMAGE_SIZE
assert LOADER_IMAGE_SIZE == MODEL_H, (
    f"Loader IMAGE_SIZE={LOADER_IMAGE_SIZE} != model image_size={MODEL_H}. "
    f"The pipeline_a loader was pinned to MCG-NJU/videomae-base; verify the pinned model_id."
)

# Trustworthy model-load check: download the Kinetics-finetuned state
# dict separately and confirm every non-classifier key matches the
# loaded model exactly (shape + presence). Classifier keys are allowed
# to differ — that is what num_labels=2 + ignore_mismatched_sizes=True
# is for. Any backbone missing / unexpected / shape-mismatch is a
# hard fail.
print()
print("Backbone integrity check (key-by-key vs Kinetics-finetuned state dict):")

def _is_classifier_key(name: str) -> bool:
    n = name.lower()
    return ("classifier" in n
            or n.endswith(".fc.weight")
            or n.endswith(".fc.bias"))

pretrained_state_dict = None
for fname in ("pytorch_model.bin", "model.safetensors"):
    try:
        ckpt_path = hf_hub_download(PRIMARY_MODEL_ID, filename=fname)
        if fname.endswith(".safetensors"):
            pretrained_state_dict = st.load_file(ckpt_path)
        else:
            pretrained_state_dict = torch.load(ckpt_path, map_location="cpu")
        break
    except Exception as exc:
        print(f"  could not fetch {fname}: {type(exc).__name__}")
        continue

if pretrained_state_dict is None:
    raise RuntimeError(
        "Backbone integrity check FAILED: could not download the Kinetics-finetuned "
        "state dict for key-by-key comparison. Check huggingface_hub / safetensors / "
        "network reachability, then re-run."
    )

model_state = dict(model.state_dict())
missing = [k for k in pretrained_state_dict if not _is_classifier_key(k) and k not in model_state]
unexpected = [k for k in model_state if not _is_classifier_key(k) and k not in pretrained_state_dict]
shape_mismatch = [
    (k, tuple(pretrained_state_dict[k].shape), tuple(model_state[k].shape))
    for k in pretrained_state_dict
    if not _is_classifier_key(k) and k in model_state
    and tuple(pretrained_state_dict[k].shape) != tuple(model_state[k].shape)
]

assert not missing, (
    f"Backbone integrity FAILED: {len(missing)} keys in Kinetics checkpoint "
    f"missing from loaded model. First 5: {missing[:5]}"
)
assert not unexpected, (
    f"Backbone integrity FAILED: {len(unexpected)} keys in loaded model "
    f"not in Kinetics checkpoint. First 5: {unexpected[:5]}"
)
assert not shape_mismatch, (
    f"Backbone integrity FAILED: {len(shape_mismatch)} keys with shape mismatch. "
    f"First 3: {shape_mismatch[:3]}"
)

backbone_params = sum(p.numel() for n, p in model.named_parameters() if not _is_classifier_key(n))
classifier_params = sum(p.numel() for n, p in model.named_parameters() if _is_classifier_key(n))
print(f"  backbone parameters  : {backbone_params:,}")
print(f"  classifier parameters: {classifier_params:,}")
print(f"  backbone keys verified: {sum(1 for k in pretrained_state_dict if not _is_classifier_key(k))}")
print(f"  classifier keys (allowed to differ): {sum(1 for k in pretrained_state_dict if _is_classifier_key(k))}")
assert backbone_params > 50_000_000, (
    f"Backbone param count {backbone_params:,} suspiciously low for VideoMAE-base."
)
assert model_cfg.num_labels == 2, f"Model num_labels is {model_cfg.num_labels}, expected 2."

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"\nModel moved to device : {device}")
print("Backbone integrity check: PASSED.")


## 4. Torch Dataset wrapping `pipeline_a/load_crops.py`

Wraps the Issue 005 Step 1 loader in a `torch.utils.data.Dataset`. The wrapper does NOT re-normalise — the loader already emits ImageNet-normalised `(T, 3, H, W) float32` tensors, and `VideoMAEForVideoClassification` consumes them directly.

The wrapper handles the 32 → 16 temporal reconciliation via uniform subsampling, driven by the model's `num_frames` config attribute (NOT a hard-coded constant — if a future VideoMAE variant expects 32 frames, the same wrapper adapts).

Each `__init__` call also runs the **real-crop label** through `label_window_from_crop_meta(...)` (the only supported real-crop entry point per the Issue 006 carry-forwards). Loading + labelling happens once per clip and is cached on the instance — the tiny-slice overfit only touches a handful of clips.

In [ ]:
import numpy as np
from typing import Sequence

from pipeline_a import (
    ALLOWED_T,
    IMAGE_SIZE,
    LoadedClip,
    InvalidClipLengthError,
    load_clip_tensor_from_shards,
)
from data.urfd_labels import (
    CSVLabels,
    WINDOW_LABEL_FALL,
    WINDOW_LABEL_NO_FALL,
    WINDOW_LABEL_UNLABELED,
    label_window_from_crop_meta,
)

def uniform_temporal_subsample(frames: np.ndarray, target_T: int) -> np.ndarray:
    """Reconcile a clip's frame count to ``target_T`` via uniform subsampling.

    ``frames`` has shape ``(T, C, H, W)`` — the pipeline_a loader's contract.
    If ``T == target_T`` the array is returned unchanged. Otherwise we
    pick ``target_T`` evenly-spaced indices in ``[0, T - 1]``.
    """
    T = frames.shape[0]
    if T == target_T:
        return frames
    if T < target_T:
        raise ValueError(f"Cannot upsample {T} frames to {target_T}; the loader disallows pad/truncate.")
    if target_T == 1:
        return frames[:1]
    if (T - 1) % (target_T - 1) == 0:
        indices = np.arange(target_T) * ((T - 1) // (target_T - 1))
    else:
        indices = np.round(np.linspace(0.0, float(T - 1), target_T)).astype(np.int64)
    return frames[indices]

class VideoMAECropDataset(torch.utils.data.Dataset):
    """Wrap pipeline_a.load_crops.load_clip_tensor_from_shards as a torch Dataset.

    Each item is a dict with:
      - ``pixel_values``: torch.float32 tensor, shape ``(MODEL_T, 3, MODEL_H, MODEL_W)``,
        ImageNet-normalised by the loader (we do NOT re-normalise).
      - ``labels``: torch.long scalar with the window-level label
        (``0 = no_fall``, ``1 = fall``).
      - ``clip_key``, ``clip_id``: provenance, copied from the LoadedClip.
    """

    def __init__(
        self,
        clip_keys: Sequence[str],
        shard_paths: Sequence[object],
        csv_by_clip_id: dict[str, CSVLabels],
        target_T: int,
        target_H: int,
        target_W: int,
    ) -> None:
        self._clip_keys = list(clip_keys)
        self._shard_paths = list(shard_paths)
        self._csv_by_clip = dict(csv_by_clip_id)
        self._target_T = int(target_T)
        self._target_H = int(target_H)
        self._target_W = int(target_W)
        self._cache: dict[str, dict[str, object]] = {}

        skipped: list[tuple[str, str]] = []
        kept: list[str] = []
        for clip_key in self._clip_keys:
            try:
                entry = self._load_and_label(clip_key)
            except Exception as exc:  # noqa: BLE001
                skipped.append((clip_key, f"{type(exc).__name__}: {exc}"))
                continue
            if entry is None:
                skipped.append((clip_key, "unlabeled window"))
                continue
            self._cache[clip_key] = entry
            kept.append(clip_key)

        self._clip_keys = kept
        if not self._clip_keys:
            raise RuntimeError(
                "VideoMAECropDataset has no usable clips after labelling. "
                f"Skipped: {skipped}"
            )
        self._skipped = tuple(skipped)

    @property
    def skipped(self) -> tuple[tuple[str, str], ...]:
        return self._skipped

    def __len__(self) -> int:
        return len(self._clip_keys)

    def _load_and_label(self, clip_key: str) -> dict[str, object] | None:
        loaded: LoadedClip | None = None
        last_exc: Exception | None = None
        for T in ALLOWED_T:
            try:
                loaded = load_clip_tensor_from_shards(
                    self._shard_paths, clip_key=clip_key, T=T,
                )
                break
            except InvalidClipLengthError as exc:
                last_exc = exc
                continue
        if loaded is None:
            if last_exc is not None:
                raise last_exc
            raise RuntimeError(f"Could not load clip {clip_key!r} at any ALLOWED_T.")

        csv = self._csv_by_clip.get(loaded.clip_id)
        if csv is None:
            raise RuntimeError(
                f"No CSVLabels registered for clip_id {loaded.clip_id!r}. "
                f"Check that the CSV directory has both URFD label files."
            )

        label_str, is_confuser = label_window_from_crop_meta(
            loaded.clip_id, list(loaded.frame_indices), csv,
        )
        if label_str == WINDOW_LABEL_UNLABELED:
            return None
        if label_str == WINDOW_LABEL_FALL:
            label_int = 1
        elif label_str == WINDOW_LABEL_NO_FALL:
            label_int = 0
        else:
            raise RuntimeError(f"Unknown label_str {label_str!r} from label_window_from_crop_meta")

        tensor = uniform_temporal_subsample(loaded.tensor, self._target_T)
        if tensor.shape[1] != 3 or tensor.shape[2] != self._target_H or tensor.shape[3] != self._target_W:
            raise RuntimeError(
                f"Clip {clip_key!r}: tensor shape {tensor.shape} != expected "
                f"(T={self._target_T}, 3, {self._target_H}, {self._target_W}). "
                f"Pipeline A loader pinned at H=W={IMAGE_SIZE}; model expects {self._target_H}. "
                f"Mismatched pinning — verify the model id and the loader's constants."
            )

        return {
            "pixel_values": torch.from_numpy(tensor),
            "labels": torch.tensor(label_int, dtype=torch.long),
            "clip_key": clip_key,
            "clip_id": loaded.clip_id,
            "is_confuser": bool(is_confuser),
        }

    def __getitem__(self, idx: int) -> dict[str, object]:
        clip_key = self._clip_keys[idx]
        return self._cache[clip_key]

## 5. Real-crop labelling through `label_window_from_crop_meta`

Load the two URFD label CSVs once and build the `clip_id → CSVLabels` map the dataset uses. Every window's label is computed via `label_window_from_crop_meta(...)` — the supported real-crop entry point — and the primitive `label_window(...)` is never used here.

In [ ]:
from data.urfd_labels import parse_urfd_csv_label_file

fall_csv = parse_urfd_csv_label_file(CSV_LABEL_DIR / FALL_CSV_FILENAME)
adl_csv = parse_urfd_csv_label_file(CSV_LABEL_DIR / ADL_CSV_FILENAME)

print(f"Fall CSV sequences    : {len(fall_csv.sequences())}    (expected 30)")
print(f"ADL  CSV sequences    : {len(adl_csv.sequences())}    (expected 40)")

csv_by_clip_id: dict[str, CSVLabels] = {}
for clip_id in unique_clip_ids:
    if "fall-" in clip_id:
        csv_by_clip_id[clip_id] = fall_csv
    elif "adl-" in clip_id:
        csv_by_clip_id[clip_id] = adl_csv
    else:
        raise SystemExit(f"Unknown clip_id family: {clip_id!r}")

print(f"clip_id -> CSV map size : {len(csv_by_clip_id)}    (expected 70)")

## 6. Stratified source-clip split (never by window)

Window-level splits leak across consecutive frames of the same fall. We split at the **clip** level (each clip's windows land in the same split), AND we stratify by source family so the validation set has at least one fall clip.

Algorithm:

1. Partition clip_ids into fall-* and adl-* families.
2. Within each family, deterministic shuffled split (seed=42) at 70/15/15.
3. Concatenate the per-family splits.

The debug-tier clip-id assertion (no clip_id lacking `*-debug-*`) keeps the data manifest free of vault data; `ExecutionContext.EVALUATION` denies frozen access for downstream eval calls.


In [ ]:
import random
from collections import defaultdict

SPLIT_SEED = 42
SPLIT_TRAIN = 0.70
SPLIT_VAL = 0.15

windows_by_clip: dict[str, list[tuple[str, str, str]]] = defaultdict(list)
for ck, ci, lbl in windows:
    windows_by_clip[ci].append((ck, ci, lbl))

all_clip_ids = sorted(windows_by_clip.keys())

# Stratified split: partition by source family, split within each.
def _is_fall_source(cid: str) -> bool:
    return "-fall-" in cid

fall_clip_ids = sorted(cid for cid in all_clip_ids if _is_fall_source(cid))
adl_clip_ids = sorted(cid for cid in all_clip_ids if not _is_fall_source(cid))

def _stratified_split(clip_ids: list[str], frac_train: float, frac_val: float, seed: int):
    rng = random.Random(seed)
    perm = list(clip_ids)
    rng.shuffle(perm)
    n = len(perm)
    n_train = max(1, int(round(n * frac_train)))
    n_val = max(1, int(round(n * frac_val)))
    if n_train + n_val >= n:
        n_val = max(1, n - n_train - 1)
    return perm[:n_train], perm[n_train:n_train + n_val], perm[n_train + n_val:]

fall_train, fall_val, fall_held = _stratified_split(fall_clip_ids, SPLIT_TRAIN, SPLIT_VAL, SPLIT_SEED)
adl_train, adl_val, adl_held = _stratified_split(adl_clip_ids, SPLIT_TRAIN, SPLIT_VAL, SPLIT_SEED)

train_clip_ids = set(fall_train + adl_train)
val_clip_ids = set(fall_val + adl_val)
held_clip_ids = set(fall_held + adl_held)

def _windows_for(clips: set[str]) -> list[tuple[str, str, str]]:
    out: list[tuple[str, str, str]] = []
    for ci in clips:
        out.extend(windows_by_clip[ci])
    return out

train_windows = _windows_for(train_clip_ids)
val_windows = _windows_for(val_clip_ids)
held_windows = _windows_for(held_clip_ids)

# Debug-tier clip-id assertion — manifest-free.
all_crop_clip_ids = {r["clip_id"] for r in records}
non_debug = sorted(cid for cid in all_crop_clip_ids if "-debug-" not in cid)
assert not non_debug, (
    f"Debug-tier assertion failed: {len(non_debug)} clip_id(s) from the crop "
    f"shards do not look like debug-tier data. Sample: {non_debug[:3]}"
)

# Window-level label distribution per split via label_window_from_crop_meta.
# True window-level fall / no_fall counts so the split is characterised
# by what each set actually contains (not the source-clip label).
def _window_label_dist(rs: list[tuple[str, str, str]]) -> tuple[int, int, int]:
    fall = no_fall = unlabeled = 0
    for ck, ci, _ in rs:
        r = next(rr for rr in records if rr["clip_key"] == ck)
        csv = fall_csv if _is_fall_source(ci) else adl_csv
        label_str, _ = label_window_from_crop_meta(ci, list(r["frame_indices"]), csv)
        if label_str == WINDOW_LABEL_FALL:
            fall += 1
        elif label_str == WINDOW_LABEL_NO_FALL:
            no_fall += 1
        else:
            unlabeled += 1
    return fall, no_fall, unlabeled

train_f, train_nf, train_ul = _window_label_dist(train_windows)
val_f, val_nf, val_ul = _window_label_dist(val_windows)
held_f, held_nf, held_ul = _window_label_dist(held_windows)

print(f"Clips total           : {len(all_clip_ids)}")
print(f"  fall-source clips   : {len(fall_clip_ids)}")
print(f"  adl-source clips    : {len(adl_clip_ids)}")
print(f"  train clip ids      : {len(train_clip_ids)}  (fall={len(fall_train)}, adl={len(adl_train)})")
print(f"  val   clip ids      : {len(val_clip_ids)}  (fall={len(fall_val)}, adl={len(adl_val)})")
print(f"  held clip ids       : {len(held_clip_ids)}  (fall={len(fall_held)}, adl={len(adl_held)})")
print("Windows (window-level via label_window_from_crop_meta):")
print(f"  train: fall={train_f}  no_fall={train_nf}  unlabeled={train_ul}")
print(f"  val:   fall={val_f}  no_fall={val_nf}  unlabeled={val_ul}")
print(f"  held:  fall={held_f}  no_fall={held_nf}  unlabeled={held_ul}")
print(f"Debug-tier assertion  : PASSED ({len(all_crop_clip_ids)} unique clip_ids, all match `*-debug-*`)")
print(f"Stratified split      : PASSED (validation has {val_f} fall windows; primary metric is val AUPRC).")
assert val_f >= 1, f"Stratified split failed: validation has {val_f} fall windows (need >= 1)"


## 7. Tiny-slice overfit

Pick a handful of fall + no-fall windows from the train split, run a few AdamW steps, and assert that the model path is wired correctly. We deliberately avoid an **absolute** loss threshold because it depends on random initial weights and the label mix — a **relative** decrease is what actually proves the model is learning.

Pass criteria (both must hold; a failure is a wiring problem, not a model-capacity problem):

1. Every step's loss is **finite** (no NaN / Inf — usually an LR / dtype / label-mismatch symptom).
2. The final avg loss is at most **50%** of the initial avg loss (substantial decrease).

We pick **3 fall + 3 no-fall** windows so the overfit has enough signal to memorise without burning compute, and we run **5 steps** with lr=5e-5. The expected pattern: initial loss near `log(2) ≈ 0.69` for a 2-class head, final loss well under half of that within a few steps.

In [ ]:
import math

TINY_FALL = 3
TINY_NOFALL = 3
TINY_STEPS = 5
TINY_LR = 5e-5
TINY_LOSS_RATIO = 0.5   # final must be <= 50% of initial

tiny_falls = [w for w in train_windows if w[2] == "fall"][:TINY_FALL]
tiny_nofalls = [w for w in train_windows if w[2] == "no_fall"][:TINY_NOFALL]
tiny_slice = tiny_falls + tiny_nofalls
tiny_clip_keys = [w[0] for w in tiny_slice]

print(f"Tiny-slice clip keys  : {len(tiny_slice)}")
for w in tiny_slice:
    print(f"  {w[0]:<44}  clip_id={w[1]:<32}  metadata-label={w[2]}")

assert len(tiny_falls) >= 1, "Need at least one fall window for the overfit test."
assert len(tiny_nofalls) >= 1, "Need at least one no-fall window for the overfit test."

tiny_dataset = VideoMAECropDataset(
    clip_keys=tiny_clip_keys,
    shard_paths=shard_paths,
    csv_by_clip_id=csv_by_clip_id,
    target_T=MODEL_T,
    target_H=MODEL_H,
    target_W=MODEL_W,
)

print(f"\nDataset size          : {len(tiny_dataset)}")
print(f"Skipped at __init__   : {len(tiny_dataset.skipped)}")

probe = tiny_dataset[0]
print("\nProbe item:")
print(f"  pixel_values shape   : {probe['pixel_values'].shape}    (expected ({MODEL_T}, 3, {MODEL_H}, {MODEL_W}))")
print(f"  pixel_values dtype   : {probe['pixel_values'].dtype}    (expected torch.float32)")
print(f"  pixel_values range   : [{probe['pixel_values'].min():.3f}, {probe['pixel_values'].max():.3f}]    (ImageNet-normalised => ~[-2.1, 2.6])")
print(f"  labels               : {int(probe['labels'])}    (0=no_fall, 1=fall)")
print(f"  clip_id              : {probe['clip_id']}")

assert probe['pixel_values'].shape == (MODEL_T, 3, MODEL_H, MODEL_W), (
    f"Pixel-value shape {probe['pixel_values'].shape} != expected ({MODEL_T}, 3, {MODEL_H}, {MODEL_W})"
)

optimizer = torch.optim.AdamW(model.parameters(), lr=TINY_LR)
model.train()
step_losses: list[float] = []
for step in range(TINY_STEPS):
    per_step_losses: list[float] = []
    for i in range(len(tiny_dataset)):
        item = tiny_dataset[i]
        pixel_values = item["pixel_values"].unsqueeze(0).to(device)
        labels = item["labels"].unsqueeze(0).to(device)
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss at step={step} item={i}: {loss.item()}")
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        per_step_losses.append(float(loss.detach().cpu().item()))
    avg = sum(per_step_losses) / len(per_step_losses)
    step_losses.append(avg)
    print(f"  step {step+1}/{TINY_STEPS} avg_loss = {avg:.4f}")

final_loss = step_losses[-1]
initial_loss = step_losses[0]
print(f"\nInitial avg loss      : {initial_loss:.4f}")
print(f"Final   avg loss      : {final_loss:.4f}")

# Pass criteria:
#   1. every loss is finite (caught per-step above; torch.isfinite is
#      used on the per-step tensor loss inside the training loop)
#   2. final loss is at most TINY_LOSS_RATIO × initial loss
#
# initial_loss and final_loss are Python floats — torch.isfinite on a
# Python float raises TypeError. Use math.isfinite for the float check.
if not (math.isfinite(initial_loss) and math.isfinite(final_loss)):
    raise RuntimeError(
        f"Tiny-slice overfit FAILED: non-finite loss. "
        f"initial={initial_loss}, final={final_loss}. "
        f"This is a WIRING problem (NaN / Inf usually = LR too high, "
        f"label mismatch, or dtype issue)."
    )
if final_loss > TINY_LOSS_RATIO * max(initial_loss, 1e-9):
    raise RuntimeError(
        f"Tiny-slice overfit FAILED: final avg loss {final_loss:.4f} did not "
        f"decrease substantially from initial {initial_loss:.4f} "
        f"(ratio {final_loss / max(initial_loss, 1e-9):.2f} > threshold {TINY_LOSS_RATIO:.2f}). "
        f"This is a WIRING problem (data pipeline / labels / head / optimiser / "
        f"tensor shape), not a model-capacity problem. Inspect the per-step losses above."
    )
print("Tiny-slice overfit    : PASSED (loss is finite and decreased substantially).")

## 8. Validation baselines (majority + frozen Kinetics + untrained head)

Before any training, we record two baselines on the validation set so the trained model can be compared against them:

1. **Majority-class all-no_fall** — every prediction is `0.0`. AUPRC is the area under the precision-recall curve; an all-zero predictor yields AUPRC equal to the positive rate (≈ 0.17 on the current val set with the stratified split). A trained model that fails to beat this baseline has not learned.
2. **Frozen Kinetics backbone + untrained binary head** — the freshly-loaded Kinetics backbone with the freshly-reinitialised 2-class head, run in `eval()` mode. Untrained head yields essentially random logits; AUPRC near 0.5 on a balanced slice.

Both baselines route predictions through the same `evaluation.metrics.classification.compute_classification_metrics` harness the trained model uses — the comparison is apples-to-apples. The majority baseline demonstrates the harness is wired correctly (no-fall-confusion); the untrained-head baseline shows the **head must learn** (Kinetics pre-training alone does not give us fall detection on URFD).


In [ ]:
# Re-load the model fresh for production training — the §7 tiny-slice
# overfit cell may have updated the classifier head. We deliberately
# do NOT reuse the §7 model.
import json
from transformers import VideoMAEForVideoClassification
from data.manifests import ClipRole, FallLabel
from evaluation.contracts import ClipLabel, ClipPrediction
from evaluation.metrics.classification import compute_classification_metrics
from evaluation.not_available import NotAvailable

model = VideoMAEForVideoClassification.from_pretrained(
    PRIMARY_MODEL_ID,
    num_labels=2,
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    ignore_mismatched_sizes=True,
).to(device)
model.eval()

# Build val dataset once and reuse through the rest of the notebook.
val_dataset = VideoMAECropDataset(
    clip_keys=[w[0] for w in val_windows],
    shard_paths=shard_paths,
    csv_by_clip_id=csv_by_clip_id,
    target_T=MODEL_T,
    target_H=MODEL_H,
    target_W=MODEL_W,
)

# Window-level labels for val (so we can use the same harness as the
# trained-model section).
def _val_window_labels() -> dict[str, int]:
    out: dict[str, int] = {}
    for ck, ci, _ in val_windows:
        r = next(rr for rr in records if rr["clip_key"] == ck)
        csv = fall_csv if _is_fall_source(ci) else adl_csv
        ls, _ = label_window_from_crop_meta(ci, list(r["frame_indices"]), csv)
        out[ck] = 1 if ls == WINDOW_LABEL_FALL else 0  # unlabeled -> no_fall (training convention)
    return out

val_window_y = _val_window_labels()

def _print_primary_window(reports, label):
    print(f"\n{label} (window-level, primary: AUPRC + recall):")
    for r in reports:
        if r.slice_key is None:
            for m in r.metrics:
                if m.name in ("auprc", "recall", "accuracy", "precision", "f1", "auc_roc", "specificity"):
                    v = m.value
                    if isinstance(v, NotAvailable):
                        print(f"  {m.name:<10} : n/a ({v.reason})")
                    else:
                        marker = "  [PRIMARY]" if m.name in ("auprc", "recall") else ""
                        print(f"  {m.name:<10} : {v:.4f}{marker}")
            cm = r.confusion_matrix
            print(f"  confusion  : tn={cm.tn} fp={cm.fp} fn={cm.fn} tp={cm.tp}")
            break

# --- Baseline 1: majority-class all-no_fall ---
all_zero_preds = [
    ClipPrediction(
        clip_id=ck, score=0.0,
        model_id="baseline_majority", dataset="urfd", role=ClipRole.VALIDATE,
    )
    for ck, _, _ in val_windows
]
all_zero_labels = [
    ClipLabel(
        clip_id=ck,
        label=(FallLabel.FALL if val_window_y[ck] == 1 else FallLabel.NO_FALL),
        dataset="urfd", role=ClipRole.VALIDATE, source_path="",
    )
    for ck, _, _ in val_windows
]
baseline_majority_reports = compute_classification_metrics(all_zero_preds, all_zero_labels)
_print_primary_window(baseline_majority_reports, "Baseline 1 (majority-class, all no_fall)")

# --- Baseline 2: frozen Kinetics + untrained binary head ---
untrained_preds = []
untrained_labels = []
with torch.no_grad():
    for i in range(len(val_dataset)):
        item = val_dataset[i]
        pixel_values = item["pixel_values"].unsqueeze(0).to(device)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=bf16_supported):
            outputs = model(pixel_values=pixel_values)
        probs = torch.softmax(outputs.logits.float(), dim=-1)[0].detach().cpu().numpy()
        fall_score = float(probs[1])
        clip_key = str(item["clip_key"])
        untrained_preds.append(ClipPrediction(
            clip_id=clip_key, score=fall_score,
            model_id="untrained_kinetics", dataset="urfd", role=ClipRole.VALIDATE,
        ))
        untrained_labels.append(ClipLabel(
            clip_id=clip_key,
            label=(FallLabel.FALL if val_window_y[clip_key] == 1 else FallLabel.NO_FALL),
            dataset="urfd", role=ClipRole.VALIDATE, source_path="",
        ))
baseline_untrained_reports = compute_classification_metrics(untrained_preds, untrained_labels)
_print_primary_window(baseline_untrained_reports, "Baseline 2 (frozen Kinetics + untrained binary head)")

# Persist a short baseline summary for the run log.
BASELINE_RESULTS = {
    "val_window_count": len(val_windows),
    "val_fall_count": val_f,
    "val_nofall_count": val_nf,
    "baseline_majority": {
        n: (m.value if not isinstance(m.value, NotAvailable) else f"n/a ({m.value.reason})")
        for r in baseline_majority_reports if r.slice_key is None
        for n, m in [(m.name, m) for m in r.metrics]
        if m.name in ("auprc", "recall", "accuracy")
    },
    "baseline_untrained_kinetics": {
        n: (m.value if not isinstance(m.value, NotAvailable) else f"n/a ({m.value.reason})")
        for r in baseline_untrained_reports if r.slice_key is None
        for n, m in [(m.name, m) for m in r.metrics]
        if m.name in ("auprc", "recall", "accuracy")
    },
}
print("\nBaseline summary recorded for the run log:")
print(json.dumps(BASELINE_RESULTS, indent=2, sort_keys=True, default=str))


## 11. Validation evaluation + persistence + FPS

We load the **best** linear-head checkpoint (by val AUPRC) and run a full evaluation pass through `evaluation.metrics.classification.compute_classification_metrics`. Predictions come from `softmax(logits)[:, 1]` (the fall probability). Clip-level scores are aggregated from window-level scores by **max-pool** (a clip is "fall" if any of its windows predicts fall); clip labels are `fall` for `fall-*` and `no_fall` for `adl-*`.

Both window-level and clip-level reports are persisted via `evaluation.result_persistence.MetricResultStore`. The store writes a versioned payload (`results.json` + `summary.txt`) under `<artifact_root>/metrics/pipeline_a/<run_id>/` with `format_version=1.0`. The `notes` field on `EvalRunMetadata` carries the model id, transformers/torch versions, checkpoint path + size, split seed, class weights, AMP dtype, peak GPU memory, and inference FPS — enough provenance that a future reader can reproduce the result.

FPS / latency is recorded for two paths: the linear head alone (cheap) and the full backbone + head pipeline (the runtime target for real-time opt in Issue 015). Model size = total parameter count + linear head parameter count + checkpoint byte size.

`EvaluationContext.EVALUATION` is used for the persistence call; frozen-unseen access is denied by design.


## 9. Frozen-backbone feature extraction + cache

We freeze the Kinetics backbone, then run `model.video_mae(pixel_values=...)` in `eval()` + `no_grad()` over every train and val window. The output's `last_hidden_state[:, 0, :]` (the `[CLS]` token) is the pooled feature the original classifier head consumes — it is the natural input to a fresh linear head.

AMP is capability-detected: `bf16` on L4/A100 where supported, `fp16` on T4. Mixed precision is enabled via `torch.amp.autocast("cuda", dtype=...)`.

Features are cached **in memory** for this prompt (240 windows × 768 hidden ≈ 740 KB — trivial). The cache is a `{features, labels, clip_keys, clip_ids, elapsed_seconds, peak_memory_bytes}` dict per split.

Peak GPU memory during extraction is captured via `torch.cuda.max_memory_allocated()`. FPS for full-backbone + head inference is recorded alongside the linear-head-only FPS in §11.


In [ ]:
import time

# Freeze the backbone. The classifier head is left trainable but we will
# throw it away and train a fresh linear head on the cached features in
# §10. Freezing here also ensures the backbone does NOT get gradient
# updates if the cache extraction ever changes.
for name, p in model.named_parameters():
    if not _is_classifier_key(name):
        p.requires_grad = False

# Capability-detected AMP dtype. Prefer fp16 on T4 (where bf16 is
# either unsupported or slow), keep bf16 on L4/A100 where supported,
# fall back to fp16 / no-AMP on older hardware.
gpu_is_t4 = "T4" in (gpu_name or "")
if gpu_is_t4:
    amp_dtype = torch.float16 if fp16_supported else None
elif bf16_supported:
    amp_dtype = torch.bfloat16
elif fp16_supported:
    amp_dtype = torch.float16
else:
    amp_dtype = None
print(f"AMP dtype: {amp_dtype}  (gpu={gpu_name}, t4={gpu_is_t4}, bf16_supported={bf16_supported}, fp16_supported={fp16_supported})")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

def _extract_features(ds):
    '''Run frozen backbone forward; return cached features + provenance.'''
    feats, labels, keys, ids = [], [], [], []
    started = time.perf_counter()
    model.eval()
    with torch.no_grad():
        for i in range(len(ds)):
            item = ds[i]
            pixel_values = item["pixel_values"].unsqueeze(0).to(device)
            with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=amp_dtype is not None):
                out = model.videomae(pixel_values=pixel_values)
                # VideoMAE has NO CLS token. The "index 0" path used by
                # the original .classifier is the first patch token (a
                # temporal tubelet). The natural feature for a fresh
                # linear head is mean-pool over all patch tokens, then
                # optional fc_norm (BatchNorm) before the head — exactly
                # the path the original classifier would take on a
                # mean-pooled input.
                hidden = out.last_hidden_state
                pooled = hidden.mean(dim=1)
                if getattr(model, "fc_norm", None) is not None:
                    pooled = model.fc_norm(pooled)
                feat = pooled.float().cpu()
            feats.append(feat)
            labels.append(int(item["labels"]))
            keys.append(str(item["clip_key"]))
            ids.append(str(item["clip_id"]))
    peak_mem = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0
    return {
        "features": torch.cat(feats, dim=0) if feats else torch.empty(0, HIDDEN_SIZE),
        "labels": torch.tensor(labels, dtype=torch.long) if labels else torch.empty(0, dtype=torch.long),
        "clip_keys": keys,
        "clip_ids": ids,
        "elapsed_seconds": time.perf_counter() - started,
        "peak_memory_bytes": peak_mem,
        "n_samples": len(labels),
    }

# Build train + val datasets and extract features.
train_dataset = VideoMAECropDataset(
    clip_keys=[w[0] for w in train_windows],
    shard_paths=shard_paths,
    csv_by_clip_id=csv_by_clip_id,
    target_T=MODEL_T,
    target_H=MODEL_H,
    target_W=MODEL_W,
)
print(f"Train dataset size: {len(train_dataset)}  (skipped at __init__: {len(train_dataset.skipped)})")
print(f"Val   dataset size: {len(val_dataset)}  (skipped at __init__: {len(val_dataset.skipped)})")

train_feats = _extract_features(train_dataset)
val_feats = _extract_features(val_dataset)
print(f"\nTrain features: {tuple(train_feats['features'].shape)}  ({train_feats['elapsed_seconds']:.2f}s, peak GPU={train_feats['peak_memory_bytes']/1e9:.2f} GB)")
print(f"Val   features: {tuple(val_feats['features'].shape)}  ({val_feats['elapsed_seconds']:.2f}s, peak GPU={val_feats['peak_memory_bytes']/1e9:.2f} GB)")


## 10. Linear head training (AdamW + checkpointing)

We train a **fresh** linear head on the cached features: `nn.Linear(HIDDEN_SIZE, 2)`. Only the head is trainable; the backbone stays frozen.

**Class weights** are inverse-frequency on the train window-level labels: `weight_pos = N / (2 * n_pos)`, `weight_neg = N / (2 * n_neg)`. This addresses the 41-fall / 199-no-fall imbalance without oversampling (cleaner for the small dataset and avoids a separate sampler).

**Optimizer**: AdamW on the head parameters only, `lr=1e-3`, `weight_decay=1e-4`. Manual training loop (`NUM_EPOCHS=30`, `BATCH_SIZE=16`).

**Checkpointing**: every epoch whose `val_auprc > best_auprc` is saved to `<artifact_root>/checkpoints/pipeline_a/linear_head.pt` with `{epoch, head_state, optimizer_state, best_val_auprc, split_seed, class_weights, hidden_size, model_id, transformers_version, torch_version}`. On re-run, `_load_latest` reads the checkpoint and resumes from `start_epoch + 1`.

**Epoch selection**: **best val AUPRC only** — never accuracy. AUPRC is the priority metric for rare fall detection; the selection rule is documented in the `best_val_auprc` checkpoint key.


In [ ]:
import torch.nn as nn
from sklearn.metrics import average_precision_score

# Fresh linear head on the cached features.
torch.manual_seed(SPLIT_SEED)
linear_head = nn.Linear(HIDDEN_SIZE, 2).to(device)
print(f"Linear head: in_features={HIDDEN_SIZE} out_features=2")
print(f"Linear head parameters: {sum(p.numel() for p in linear_head.parameters()):,}")

# Class weights: inverse frequency on train window-level labels.
n_pos = int((train_feats["labels"] == 1).sum().item())
n_neg = int((train_feats["labels"] == 0).sum().item())
n_total = n_pos + n_neg
weight_pos = n_total / (2 * max(n_pos, 1))
weight_neg = n_total / (2 * max(n_neg, 1))
class_weights = torch.tensor([weight_neg, weight_pos], dtype=torch.float32, device=device)
print(f"\nTrain class counts: pos (fall)={n_pos}  neg (no_fall)={n_neg}")
print(f"Class weights: no_fall={weight_neg:.3f}  fall={weight_pos:.3f}")

# Move cached features to device.
train_feats_X = train_feats["features"].to(device)
train_feats_y = train_feats["labels"].to(device)
val_feats_X = val_feats["features"].to(device)
val_feats_y = val_feats["labels"].to(device)

# AdamW on the head only.
optimizer = torch.optim.AdamW(linear_head.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Checkpoint / resume.
CHECKPOINT_DIR = layout.artifact_root / "checkpoints" / "pipeline_a"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = CHECKPOINT_DIR / "linear_head_best.pt"  # best by val AUPRC
latest_path = CHECKPOINT_DIR / "linear_head_latest.pt"     # most recent epoch

def _load_latest(path, head, opt):
    if not path.is_file():
        return 0, -1.0
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    head.load_state_dict(ckpt["head_state"])
    opt.load_state_dict(ckpt["optimizer_state"])
    return int(ckpt.get("epoch", -1)) + 1, float(ckpt.get("best_val_auprc", -1.0))

start_epoch, best_auprc = _load_latest(checkpoint_path, linear_head, optimizer)
print(f"\nResuming from epoch {start_epoch}, best val AUPRC so far: {best_auprc:.4f}")

NUM_EPOCHS = 30
BATCH_SIZE = 16

def _eval_auprc():
    linear_head.eval()
    with torch.no_grad():
        logits = linear_head(val_feats_X)
        probs = torch.softmax(logits, dim=-1)[:, 1]
    return float(average_precision_score(val_feats_y.cpu().numpy(), probs.cpu().numpy()))

def _save_ckpt_to(path, epoch, head, opt, best, seed, weights, hidden, mid, tver, torchver):
    torch.save({
        "epoch": epoch,
        "head_state": head.state_dict(),
        "optimizer_state": opt.state_dict(),
        "best_val_auprc": best,
        "split_seed": seed,
        "class_weights": weights,
        "hidden_size": hidden,
        "model_id": mid,
        "transformers_version": str(tver),
        "torch_version": str(torchver),
    }, path)

def _save_ckpt(epoch, head, opt, best, seed, weights, hidden, mid, tver, torchver):
    # Backward-compat wrapper: writes to the best-checkpoint path.
    _save_ckpt_to(checkpoint_path, epoch, head, opt, best, seed, weights, hidden, mid, tver, torchver)

for epoch in range(start_epoch, NUM_EPOCHS):
    linear_head.train()
    perm = torch.randperm(len(train_feats_X), device=device)
    train_loss = 0.0
    n_batches = 0
    for i in range(0, len(perm), BATCH_SIZE):
        idx = perm[i:i + BATCH_SIZE]
        x = train_feats_X[idx]
        y = train_feats_y[idx]
        logits = linear_head(x)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += float(loss.detach().cpu().item())
        n_batches += 1
    train_loss /= max(n_batches, 1)
    val_auprc = _eval_auprc()
    is_best = val_auprc > best_auprc
    # Always save the latest checkpoint (for resume on crash).
    _save_ckpt_to(latest_path, epoch, linear_head, optimizer, best_auprc,
                SPLIT_SEED, [float(weight_neg), float(weight_pos)],
                HIDDEN_SIZE, PRIMARY_MODEL_ID,
                transformers.__version__, torch.__version__)
    if is_best:
        best_auprc = val_auprc
        _save_ckpt_to(checkpoint_path, epoch, linear_head, optimizer, best_auprc,
                    SPLIT_SEED, [float(weight_neg), float(weight_pos)],
                    HIDDEN_SIZE, PRIMARY_MODEL_ID,
                    transformers.__version__, torch.__version__)
    print(f"  epoch {epoch+1:>2}/{NUM_EPOCHS}  train_loss={train_loss:.4f}  val_auprc={val_auprc:.4f}  best={best_auprc:.4f}{'  *' if is_best else ''}")

print(f"\nBest val AUPRC: {best_auprc:.4f}")
print(f"Best checkpoint:    {checkpoint_path}  ({checkpoint_path.stat().st_size} bytes)")
print(f"Latest checkpoint:   {latest_path}  ({latest_path.stat().st_size} bytes)")


In [ ]:
import time
import json as _json
from data.manifests import ClipRole, FallLabel
from evaluation.contracts import ClipLabel, ClipPrediction
from evaluation.metrics.classification import compute_classification_metrics
from evaluation.not_available import NotAvailable
from evaluation.execution_context import ExecutionContext
from evaluation.result_persistence import (
    MetricResultStore, make_default_metadata,
)

# Load the best checkpoint (by val AUPRC).
ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
linear_head.load_state_dict(ckpt["head_state"])
linear_head.eval()
print(f"Loaded best checkpoint: epoch={ckpt['epoch']}  best_val_auprc={ckpt['best_val_auprc']:.4f}")

# --- Window-level evaluation ---
with torch.no_grad():
    logits = linear_head(val_feats_X)
    val_probs = torch.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()

window_predictions = [
    ClipPrediction(
        clip_id=val_feats["clip_keys"][i], score=float(val_probs[i]),
        model_id="videomae-kinetics-linearhead",
        dataset="urfd", role=ClipRole.VALIDATE,
    )
    for i in range(len(val_probs))
]
window_labels = [
    ClipLabel(
        clip_id=val_feats["clip_keys"][i],
        label=(FallLabel.FALL if int(val_feats["labels"][i]) == 1 else FallLabel.NO_FALL),
        dataset="urfd", role=ClipRole.VALIDATE, source_path="",
    )
    for i in range(len(val_probs))
]
window_reports = compute_classification_metrics(window_predictions, window_labels)

# --- Clip-level evaluation (max-pool over window scores per clip) ---
clip_to_max_score: dict[str, float] = {}
clip_to_label_int: dict[str, int] = {}
for i, cid in enumerate(val_feats["clip_ids"]):
    score = float(val_probs[i])
    if cid not in clip_to_max_score or score > clip_to_max_score[cid]:
        clip_to_max_score[cid] = score
    if cid not in clip_to_label_int:
        clip_to_label_int[cid] = (1 if "-fall-" in cid else 0)

clip_predictions = [
    ClipPrediction(
        clip_id=cid, score=score,
        model_id="videomae-kinetics-linearhead",
        dataset="urfd", role=ClipRole.VALIDATE,
    )
    for cid, score in clip_to_max_score.items()
]
clip_labels = [
    ClipLabel(
        clip_id=cid,
        label=(FallLabel.FALL if clip_to_label_int[cid] == 1 else FallLabel.NO_FALL),
        dataset="urfd", role=ClipRole.VALIDATE, source_path="",
    )
    for cid in clip_to_max_score
]
clip_reports = compute_classification_metrics(clip_predictions, clip_labels)

def _print_full(reports, label, primary=("auprc", "recall")):
    print(f"\n{label} metrics (primary: AUPRC + recall — not accuracy):")
    for r in reports:
        if r.slice_key is None:
            for m in r.metrics:
                if m.name in ("accuracy", "precision", "recall", "specificity", "f1", "auc_roc", "auprc"):
                    v = m.value
                    if isinstance(v, NotAvailable):
                        print(f"  {m.name:<12} : n/a ({v.reason})")
                    else:
                        marker = "  [PRIMARY]" if m.name in primary else ""
                        print(f"  {m.name:<12} : {v:.4f}{marker}")
            cm = r.confusion_matrix
            print(f"  confusion     : tn={cm.tn}  fp={cm.fp}  fn={cm.fn}  tp={cm.tp}")
            break

_print_full(window_reports, "Window-level")
_print_full(clip_reports, "Clip-level (max-pool)")

# --- FPS / latency / model size ---
started = time.perf_counter()
n_inf = 0
linear_head.eval()
with torch.no_grad():
    for i in range(len(val_feats_X)):
        x = val_feats_X[i:i + 1]
        _ = linear_head(x)
        n_inf += 1
linear_inf_elapsed = time.perf_counter() - started
linear_inf_fps = n_inf / max(linear_inf_elapsed, 1e-9)
print(f"\nLinear-head inference: {n_inf} samples in {linear_inf_elapsed:.2f}s = {linear_inf_fps:.1f} FPS")

started = time.perf_counter()
n_full = 0
model.eval()
linear_head.eval()
with torch.no_grad():
    for i in range(len(val_dataset)):
        item = val_dataset[i]
        pixel_values = item["pixel_values"].unsqueeze(0).to(device)
        with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=amp_dtype is not None):
            out = model.videomae(pixel_values=pixel_values)
            hidden = out.last_hidden_state
            pooled = hidden.mean(dim=1)
            if getattr(model, "fc_norm", None) is not None:
                pooled = model.fc_norm(pooled)
            feat = pooled.float()
            _ = linear_head(feat)
        n_full += 1
full_inf_elapsed = time.perf_counter() - started
full_inf_fps = n_full / max(full_inf_elapsed, 1e-9)
print(f"Full backbone+head inference: {n_full} samples in {full_inf_elapsed:.2f}s = {full_inf_fps:.1f} FPS")

n_total_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_head_params = sum(p.numel() for p in linear_head.parameters())
ckpt_bytes = checkpoint_path.stat().st_size
print(f"\nModel size: total {n_total_params/1e6:.1f}M  (trainable {n_trainable/1e6:.1f}M after backbone freeze)")
print(f"Linear head: {n_head_params:,} params ({n_head_params/1e3:.1f}K)")
print(f"Linear head checkpoint: {ckpt_bytes/1024:.1f} KB")
print(f"Feature extraction peak GPU memory: {train_feats['peak_memory_bytes']/1e9:.2f} GB")

# --- Versioned result persistence via evaluation.result_persistence ---
all_metric_results = []
for r in window_reports + clip_reports:
    all_metric_results.extend(r.metric_results())

ckpt_id = f"ep{ckpt.get('epoch', '?')}_auprc{ckpt.get('best_val_auprc', 0.0):.4f}"
run_id = f"006_p2_kinetics_{ckpt_id}"
metadata = make_default_metadata(
    run_id=run_id,
    model_id=PRIMARY_MODEL_ID,
    context=ExecutionContext.EVALUATION,
    notes=_json.dumps({
        "transformers_version": transformers.__version__,
        "torch_version": torch.__version__,
        "model_total_params": n_total_params,
        "linear_head_params": n_head_params,
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_bytes": ckpt_bytes,
        "split_seed": SPLIT_SEED,
        "class_weights": [float(weight_neg), float(weight_pos)],
        "amp_dtype": str(amp_dtype),
        "feature_extraction_seconds_train": train_feats["elapsed_seconds"],
        "feature_extraction_seconds_val": val_feats["elapsed_seconds"],
        "feature_extraction_peak_memory_gb": train_feats["peak_memory_bytes"] / 1e9,
        "linear_head_inference_fps": linear_inf_fps,
        "full_backbone_head_inference_fps": full_inf_fps,
        "best_val_auprc": ckpt.get("best_val_auprc", 0.0),
        "best_epoch": ckpt.get("epoch", 0),
        "baseline_majority": BASELINE_RESULTS["baseline_majority"],
        "baseline_untrained_kinetics": BASELINE_RESULTS["baseline_untrained_kinetics"],
        "val_window_count": BASELINE_RESULTS["val_window_count"],
        "val_fall_count": BASELINE_RESULTS["val_fall_count"],
        "val_nofall_count": BASELINE_RESULTS["val_nofall_count"],
    }, sort_keys=True, default=str),
)

store = MetricResultStore(METRICS_DIR)
try:
    saved_dir = store.save(metadata, all_metric_results, overwrite=False)
    print(f"\nVersioned result JSON saved to: {saved_dir}")
except FileExistsError:
    print(f"Result run_id={run_id!r} already exists; overwriting.")
    saved_dir = store.save(metadata, all_metric_results, overwrite=True)
    print(f"Overwrote: {saved_dir}")

# Sanity: confirm result JSON's primary metrics were recorded (not N/A).
def _find(reports, name):
    for r in reports:
        if r.slice_key is None:
            for m in r.metrics:
                if m.name == name:
                    return m.value
    return None

clip_auprc = _find(clip_reports, "auprc")
clip_recall = _find(clip_reports, "recall")
assert not isinstance(clip_auprc, NotAvailable), (
    "Clip-level AUPRC is NotAvailable; the harness did not produce a numeric reading."
)
assert not isinstance(clip_recall, NotAvailable), (
    "Clip-level recall is NotAvailable; the harness did not produce a numeric reading."
)
print("\nFinal primary metrics (clip-level, max-pool):")
print(f"  AUPRC  : {float(clip_auprc):.4f}  [PRIMARY]")
print(f"  recall : {float(clip_recall):.4f}  [PRIMARY]")
print("\nNote: AUPRC + recall are the primary metrics for this evaluation. Accuracy is intentionally not used for selection or headline reporting.")


## 12. Done — Issue 006 Prompt #2 scope

What this prompt proved:

- **Trustworthy model load**: `MCG-NJU/videomae-base-finetuned-kinetics` loaded with a 2-class head; backbone integrity verified key-by-key against the Kinetics-finetuned state dict. No backbone missing / unexpected / shape-mismatched keys; only the classifier head is allowed to differ.
- **Stratified source-clip split** keeps fall support in every split (asserted via the per-split window-level label distribution via `label_window_from_crop_meta`).
- **Baselines on validation** (majority-class, frozen Kinetics + untrained head) recorded before any training; the trained model must beat them on AUPRC.
- **Frozen-backbone feature extraction** with capability-detected AMP (bf16 on L4/A100, fp16 on T4), `eval()` + `no_grad()`, peak GPU memory recorded, features cached in memory.
- **Linear head training** with weighted cross-entropy (inverse-frequency), AdamW, manual loop. Best epoch selected by **val AUPRC only** — never accuracy.
- **Checkpointing + resume**: every epoch with `val_auprc > best` writes `<artifact_root>/checkpoints/pipeline_a/linear_head.pt`; on re-run the latest checkpoint is loaded and training resumes. The checkpointer writes `{epoch, head_state, optimizer_state, best_val_auprc, split_seed, class_weights, hidden_size, model_id, transformers_version, torch_version}`.
- **Validation evaluation** through `evaluation.metrics.classification.compute_classification_metrics` on both window-level and clip-level (max-pool). Predictions come from `softmax(logits)[:, 1]`; clip labels are `fall` for `fall-*` and `no_fall` for `adl-*`. AUPRC and recall are the **primary** metrics — accuracy is intentionally not the primary.
- **Versioned result JSON** persisted via `evaluation.result_persistence.MetricResultStore`; metadata carries model id, transformers/torch versions, checkpoint path + size, split seed, class weights, AMP dtype, peak GPU memory, and inference FPS.
- **FPS / latency / model size** recorded for the linear head alone and for full backbone+head.
- **Frozen unseen data closed**: every eval call uses `ExecutionContext.EVALUATION`, which denies frozen access by design.
- **No committed weights / checkpoints / features**: the linear-head checkpoint lives under `<artifact_root>/checkpoints/pipeline_a/`, which `.gitignore` already excludes (`artifacts/checkpoints/`). The result JSON is the only artefact persisted; the checkpoint is regenerated by re-running the notebook.

What is **not** in this prompt and is **deferred**:

- Full fine-tune (unfreezing the backbone) + scheduler + LR warmup + early stopping.
- Class-imbalance handling beyond inverse-frequency class weights (no oversampling / focal loss).
- Frozen-unseen test set evaluation (out of scope at this prompt; `ExecutionContext.FINAL_JUDGEMENT` will be required when it's wired).
- Pipeline C fusion / post-verification.
- Plain `MCG-NJU/videomae-base` ablation (the note is in §3; not the main path).
- Real-time inference optimisation (Issue 015).

If you want to extend this: bump `NUM_EPOCHS` for a longer fine-tune, or add a small LR-warmup scheduler. The wiring is in place; only the training-loop knobs need tuning.
